# 家庭用电数据预处理与日级数据集构造

本 Notebook 完整实现以下流程：

1. 读取分钟级用电数据，将 `?` 等标记识别为缺失值；
2. 合并日期和时间，检查重复或缺失的分钟时间戳；
3. 统计各列缺失以及缺失是否同步发生；
4. 按变量含义进行日级聚合；
5. 对日总量按有效分钟比例修正；
6. 将覆盖率低于 80% 的日期标记为低质量日；
7. 删除首尾不完整日期；
8. 依次使用“同月份+同星期几中位数、时间插值、前向/后向填充”补全低质量日；
9. 计算 `sub_metering_remainder`；
10. 合并 Paris-Montsouris 月度天气信息，并仅保留指定变量；
11. 检查缺失值、负值、日期连续性并保存完整日级数据集。

最终结果不在这里划分训练集、验证集或测试集。模型训练时应再按照时间顺序划分。

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd()
RAW_POWER = ROOT / "data" / "raw" / "household_power_consumption.txt"
DAILY_WEATHER = ROOT / "data" / "intermediate" / "weather_daily_repeated_2006-12-16_2010-11-26.csv"
FINAL_DIR = ROOT / "data" / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

MIN_COVERAGE = 0.80

RAW_COLUMNS = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

SUM_COLUMNS = [
    "Global_active_power",
    "Global_reactive_power",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

AVERAGE_COLUMNS = ["Voltage", "Global_intensity"]

print("原始用电文件：", RAW_POWER)
print("天气文件：", DAILY_WEATHER)

## 1. 读取分钟级数据并建立时间索引

`na_values` 将问号、空字符串等统一解析为缺失值。所有电力变量再显式转换成数值类型。

In [ ]:
df = pd.read_csv(
    RAW_POWER,
    sep=";",
    na_values=["?", "nan", "NaN", ""],
    low_memory=False,
)

df["datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S",
)

df = df.sort_values("datetime").set_index("datetime")

for column in RAW_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("原始记录数：", len(df))
print("起始时间：", df.index.min())
print("结束时间：", df.index.max())
display(df.head())

## 2. 检查并补齐完整分钟时间轴

这里区分两类缺失：

- 时间戳存在，但电力数值为空；
- 时间戳本身不存在。

如果缺少时间戳，`reindex` 会将其补入并留下空值，供后续统一处理。

In [ ]:
duplicate_timestamps = int(df.index.duplicated().sum())

full_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="1min",
)
missing_timestamps = full_index.difference(df.index)

print("理论分钟数：", len(full_index))
print("实际记录数：", len(df))
print("缺失时间戳数：", len(missing_timestamps))
print("重复时间戳数：", duplicate_timestamps)

if duplicate_timestamps:
    raise ValueError("数据中存在重复时间戳，请先处理。")

df = df.reindex(full_index)
df.index.name = "datetime" 

## 3. 缺失值诊断

In [ ]:
missing_count = df[RAW_COLUMNS].isna().sum()
missing_ratio_by_column = df[RAW_COLUMNS].isna().mean()
missing_any = df[RAW_COLUMNS].isna().any(axis=1)
missing_all = df[RAW_COLUMNS].isna().all(axis=1)

missing_summary = pd.DataFrame(
    {
        "missing_count": missing_count,
        "missing_ratio": missing_ratio_by_column,
    }
)

display(missing_summary)
print("存在任意数值缺失的分钟数：", int(missing_any.sum()))
print("全部电力变量同时缺失的分钟数：", int(missing_all.sum()))
print("是否为同步缺失：", bool(missing_any.equals(missing_all)))

## 4. 统计每日覆盖率

第一天与最后一天并非完整的 24 小时，因此覆盖率使用
`valid_minutes / total_minutes`，而不是直接除以 1440。

In [ ]:
valid_minutes = (
    df["Global_active_power"].notna().resample("D").sum().astype(int)
)
total_minutes = df["Global_active_power"].resample("D").size().astype(int)
coverage = valid_minutes / total_minutes

daily_quality = pd.DataFrame(
    {
        "valid_minutes": valid_minutes,
        "total_minutes": total_minutes,
        "coverage": coverage,
        "missing_ratio": 1 - coverage,
    }
)

print("包含缺失分钟的日期数：", int((coverage < 1).sum()))
print("完全缺失的日期数：", int((valid_minutes == 0).sum()))
display(daily_quality.loc[daily_quality["coverage"] < 1].head(10))

## 5. 按变量含义进行日级聚合

- 总量型变量：按日求和，并乘以 `total_minutes / valid_minutes` 修正缺失分钟导致的低估；
- 状态型变量：按有效分钟直接求日均值。

In [ ]:
daily_sum = df[SUM_COLUMNS].resample("D").sum(min_count=1)

daily_sum_corrected = daily_sum.div(
    valid_minutes.replace(0, np.nan),
    axis=0,
).mul(total_minutes, axis=0)

daily_average = df[AVERAGE_COLUMNS].resample("D").mean()

low_quality_day = coverage < MIN_COVERAGE
daily_sum_corrected.loc[low_quality_day, :] = np.nan
daily_average.loc[low_quality_day, :] = np.nan

daily_before_imputation = pd.concat(
    [daily_sum_corrected, daily_average],
    axis=1,
)
daily_before_imputation["valid_minutes"] = valid_minutes
daily_before_imputation["total_minutes"] = total_minutes
daily_before_imputation["coverage"] = coverage
daily_before_imputation["missing_ratio"] = 1 - coverage
daily_before_imputation["low_quality_flag"] = low_quality_day.astype(int)

print("覆盖率低于 80% 的日期数（包含首尾日期时）：", int(low_quality_day.sum()))

## 6. 删除首尾不完整日期

仅保留理论分钟数等于 1440 的完整日期，以免首尾日总量偏低。

In [ ]:
full_day_mask = total_minutes == 1440
daily_before_imputation = daily_before_imputation.loc[full_day_mask].copy()

print("删除的不完整日期数：", int((~full_day_mask).sum()))
print("保留完整日期数：", len(daily_before_imputation))
print("保留范围：", daily_before_imputation.index.min(), "至", daily_before_imputation.index.max())

## 7. 填补低质量日期

处理顺序：

1. 同月份、同星期几的中位数；
2. 仍缺失时按时间插值；
3. 最后使用前向和后向填充处理边界。

质量字段保留原始缺失情况，不会在填补后被改成“高质量”。

In [ ]:
daily_df = daily_before_imputation.copy()
electricity_columns = SUM_COLUMNS + AVERAGE_COLUMNS

originally_missing = daily_df[electricity_columns].isna().any(axis=1)
daily_df["_month"] = daily_df.index.month
daily_df["_weekday"] = daily_df.index.weekday

median_filled = pd.Series(False, index=daily_df.index)

for column in electricity_columns:
    was_missing = daily_df[column].isna()
    group_median = daily_df.groupby(
        ["_month", "_weekday"]
    )[column].transform("median")
    daily_df[column] = daily_df[column].fillna(group_median)
    median_filled |= was_missing & daily_df[column].notna()

remaining_before_interpolation = daily_df[electricity_columns].isna().any(axis=1)

daily_df[electricity_columns] = (
    daily_df[electricity_columns]
    .interpolate(method="time")
    .ffill()
    .bfill()
)

interpolation_filled = (
    remaining_before_interpolation
    & daily_df[electricity_columns].notna().all(axis=1)
)

daily_df = daily_df.drop(columns=["_month", "_weekday"])
daily_df["imputed_flag"] = originally_missing.astype(int)
daily_df["imputation_method"] = "none"
daily_df.loc[median_filled, "imputation_method"] = "month_weekday_median"
daily_df.loc[interpolation_filled, "imputation_method"] = "time_interpolation"

print("中位数填补日期数：", int(median_filled.sum()))
print("时间插值填补日期数：", int(interpolation_filled.sum()))
display(
    daily_df.loc[
        daily_df["imputed_flag"].eq(1),
        ["coverage", "missing_ratio", "imputation_method"],
    ]
)

## 8. 计算剩余分表能耗并统一列名

日级公式与分钟级公式一致：

`global_active_power × 1000 / 60 - sub_metering_1 - sub_metering_2 - sub_metering_3`

In [ ]:
daily_df["Sub_metering_remainder"] = (
    daily_df["Global_active_power"] * 1000 / 60
    - daily_df[
        ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
    ].sum(axis=1)
).clip(lower=0)

rename_map = {
    "Global_active_power": "global_active_power",
    "Global_reactive_power": "global_reactive_power",
    "Voltage": "voltage",
    "Global_intensity": "global_intensity",
    "Sub_metering_1": "sub_metering_1",
    "Sub_metering_2": "sub_metering_2",
    "Sub_metering_3": "sub_metering_3",
    "Sub_metering_remainder": "sub_metering_remainder",
}

daily_df = daily_df.rename(columns=rename_map)
daily_before_imputation = daily_before_imputation.rename(columns=rename_map)

## 9. 合并天气并筛选最终变量

天气数据是月度统计量，因此同一个月内每天使用相同值。合并在日级完成，避免在分钟级重复数据。

In [ ]:
weather = pd.read_csv(DAILY_WEATHER, parse_dates=["Date"])
weather = (
    weather.drop(columns=["AAAAMM"])
    .rename(columns={"Date": "date"})
    .set_index("date")
)

final_df = daily_df.join(weather, how="left", validate="1:1")

weather_columns = ["RR", "NBJRR1", "NBJRR5", "NBJRR10", "NBJBROU"]
if final_df[weather_columns].isna().any().any():
    raise ValueError("天气数据合并后出现缺失，请检查日期范围。")

final_df.insert(0, "date", final_df.index.strftime("%Y-%m-%d"))

final_columns = [
    "date",
    "global_active_power",
    "global_reactive_power",
    "voltage",
    "global_intensity",
    "sub_metering_1",
    "sub_metering_2",
    "sub_metering_3",
    "sub_metering_remainder",
    "RR",
    "NBJRR1",
    "NBJRR5",
    "NBJRR10",
    "NBJBROU",
]

final_df = final_df.reset_index(drop=True)[final_columns]

display(final_df.head())

## 10. 最终质量检查

In [ ]:
nonnegative_columns = [
    "global_active_power",
    "global_reactive_power",
    "global_intensity",
    "sub_metering_1",
    "sub_metering_2",
    "sub_metering_3",
    "sub_metering_remainder",
]

date_series = pd.to_datetime(final_df["date"])
is_daily_continuous = date_series.diff().dropna().eq(pd.Timedelta(days=1)).all()

quality_report = {
    "rows": int(len(final_df)),
    "columns": int(final_df.shape[1]),
    "date_start": final_df["date"].iloc[0],
    "date_end": final_df["date"].iloc[-1],
    "continuous_daily_index": bool(is_daily_continuous),
    "missing_cells": int(final_df.isna().sum().sum()),
    "low_quality_days": int(daily_df["low_quality_flag"].sum()),
    "imputed_days": int(daily_df["imputed_flag"].sum()),
    "negative_counts": {
        column: int((final_df[column] < 0).sum())
        for column in nonnegative_columns
    },
    "voltage_min": float(final_df["voltage"].min()),
    "voltage_max": float(final_df["voltage"].max()),
}

print(json.dumps(quality_report, ensure_ascii=False, indent=2))

assert quality_report["rows"] == 1440
assert quality_report["continuous_daily_index"]
assert quality_report["missing_cells"] == 0
assert sum(quality_report["negative_counts"].values()) == 0

## 11. 保存完整数据集

只保存完整的日级数据，不在此处划分训练、验证或测试子集。

In [ ]:
final_path = FINAL_DIR / "daily_household_power_weather.csv"

final_df.to_csv(final_path, index=False, encoding="utf-8-sig")

print("完整数据集已保存：", final_path)
print("最终列数：", len(final_df.columns))
print("最终列名：", final_df.columns.tolist())

## 使用提醒

- `global_active_power` 是预测目标，也可作为过去 90 天的输入特征；
- 最终 CSV 只保留日期、8 个用电变量和 5 个天气变量；
- 必须按照时间顺序划分训练和测试数据，不能随机打乱后再切分；
- 本 Notebook 按附件要求在完整时间线上计算“同月份+同星期几中位数”。
  如果需要最严格的无泄漏实验，应在每个训练折内重新拟合该中位数。